# Task 2 — Localization (§2.5 Object Detection)

**Kaggle setup:** Accelerator → GPU T4x2 · Internet ON

**Prerequisite:** Task 1 must be done — you need the `CLASSIFIER_DRIVE_ID` so the
pretrained encoder can be loaded.

**After this notebook you will have:**
- `checkpoints/localizer.pth` saved to Kaggle output
- W&B detection table logged for §2.5
- `LOCALIZER_DRIVE_ID` updated in `models/multitask.py` and pushed to GitHub
- Ready to submit **4.2a / 4.2b** to Gradescope

## 1 · Setup

In [ ]:
import os

os.environ["WANDB_API_KEY"] = "PASTE_YOUR_WANDB_API_KEY_HERE"

GITHUB_TOKEN    = "PASTE_YOUR_GITHUB_TOKEN_HERE"
GITHUB_USERNAME = "usnaveen"
GITHUB_REPO     = "A2_Deep_Learning"

!git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git da6401_assignment_2
%cd da6401_assignment_2

!pip install -q wandb albumentations gdown scikit-learn

import torch
print(f"CUDA available: {torch.cuda.is_available()}")

## 2 · Download Dataset

In [ ]:
from data.pets_dataset import download_oxford_pet
download_oxford_pet("./data/oxford_pet")

## 3 · Download `classifier.pth` from Google Drive

The localizer training script automatically loads the pretrained encoder from
`checkpoints/classifier.pth` if it exists.  Download it from Drive so we start
with the encoder weights you trained in Task 1.

In [ ]:
# ── Paste the CLASSIFIER_DRIVE_ID you set in Task 1 ──────────────────────────
CLASSIFIER_DRIVE_ID = "PASTE_CLASSIFIER_DRIVE_ID_HERE"
# ─────────────────────────────────────────────────────────────────────────────

import gdown, os
os.makedirs("checkpoints", exist_ok=True)
gdown.download(id=CLASSIFIER_DRIVE_ID, output="checkpoints/classifier.pth", quiet=False, fuzzy=True)
print(f"✅  classifier.pth downloaded ({os.path.getsize('checkpoints/classifier.pth')/1e6:.1f} MB)")

## 4 · §2.5 — Localization Training

Trains a bounding-box regressor on top of the pretrained VGG11 encoder.
Loss = MSE + IoU.  Logs a detection table to W&B at the end.

In [ ]:
!python train.py --task localize --experiment exp-detection --epochs 120 --lr 1e-3 --batch-size 32 --num-workers 2

## 5 · Check Best Checkpoint IoU

In [ ]:
import torch
from data.pets_dataset import create_dataloaders
from models.localization import VGG11Localizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_, _, test_loader = create_dataloaders(root="./data/oxford_pet", batch_size=64, num_workers=2)

model = VGG11Localizer(dropout_p=0.5, image_size=224).to(device)
model.load_state_dict(torch.load("checkpoints/localizer.pth", map_location=device, weights_only=False))
model.eval()

def compute_iou(pred, gt, eps=1e-6):
    px1=pred[:,0]-pred[:,2]/2; py1=pred[:,1]-pred[:,3]/2
    px2=pred[:,0]+pred[:,2]/2; py2=pred[:,1]+pred[:,3]/2
    tx1=gt[:,0]-gt[:,2]/2;     ty1=gt[:,1]-gt[:,3]/2
    tx2=gt[:,0]+gt[:,2]/2;     ty2=gt[:,1]+gt[:,3]/2
    ix1=torch.max(px1,tx1); iy1=torch.max(py1,ty1)
    ix2=torch.min(px2,tx2); iy2=torch.min(py2,ty2)
    inter=(ix2-ix1).clamp(0)*(iy2-iy1).clamp(0)
    union=(px2-px1).clamp(0)*(py2-py1).clamp(0)+(tx2-tx1).clamp(0)*(ty2-ty1).clamp(0)-inter
    return inter/(union+eps)

ious_05, ious_075, total = 0, 0, 0
with torch.no_grad():
    for batch in test_loader:
        pred = model(batch["image"].to(device)).cpu()
        gt   = batch["bbox"]
        iou  = compute_iou(pred, gt)
        ious_05  += (iou >= 0.50).sum().item()
        ious_075 += (iou >= 0.75).sum().item()
        total += len(iou)

print(f"\n✅  Accuracy @ IoU≥0.50 = {100*ious_05/total:.1f}%  (need > 60% for 4.2a)")
print(f"   Accuracy @ IoU≥0.75 = {100*ious_075/total:.1f}%  (need > 40% for 4.2b)")

## 6 · Upload `localizer.pth` to Google Drive

1. Kaggle right sidebar → **Output** → download `checkpoints/localizer.pth`
2. [drive.google.com](https://drive.google.com) → upload → **Share: Anyone with the link**
3. Copy the **FILE_ID** from the share URL and paste below

In [ ]:
LOCALIZER_DRIVE_ID = "PASTE_FILE_ID_HERE"

import re
multitask_path = "models/multitask.py"
with open(multitask_path) as f:
    content = f.read()
content = re.sub(
    r'LOCALIZER_DRIVE_ID\s*=\s*"[^"]*"',
    f'LOCALIZER_DRIVE_ID  = "{LOCALIZER_DRIVE_ID}"',
    content
)
with open(multitask_path, "w") as f:
    f.write(content)
print(f"✅  Updated models/multitask.py → LOCALIZER_DRIVE_ID = '{LOCALIZER_DRIVE_ID}'")

# Quick download sanity check
import gdown, tempfile, os
with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as tmp:
    tmp_path = tmp.name
try:
    gdown.download(id=LOCALIZER_DRIVE_ID, output=tmp_path, quiet=False, fuzzy=True)
    print(f"✅  Download test passed ({os.path.getsize(tmp_path)/1e6:.1f} MB)")
except Exception as e:
    print(f"❌  Download test FAILED: {e}")
finally:
    os.unlink(tmp_path)

## 7 · Push to GitHub → Submit to Gradescope

In [ ]:
!git config user.email "naveen@student.iitm.ac.in"
!git config user.name  "Naveen"
!git add models/multitask.py
!git diff --cached --stat
!git commit -m "task2: set LOCALIZER_DRIVE_ID for Gradescope submission"
!git push
print("\n🎉  Done! Submit to Gradescope to check 4.2a (IoU≥0.5 > 60%) and 4.2b (IoU≥0.75 > 40%)")